# Fetch News Articles — Per-Ticker News Dataset

Builds the per-ticker news dataset required for the paper replication. For each ticker:

1. Fetch all Refinitiv headlines over the paper window (2003-01-01 to 2014-12-31) using month-by-month chunking.
2. Compute coverage stats: articles per day, articles per week, and whether the stock passes the paper's news filter (≥ 1 article/week on average).
3. Fetch the full article text for every headline via `storyId`.
4. Save a Parquet dataset with one row per article: date, headline, story text, and source.

**Run this notebook once per ticker.** Set `TICKER` below and re-run. A checkpoint file is written
every 50 stories so the text-fetching step can be resumed without re-fetching from scratch.

**Requires:** LSEG Workspace running and signed in (desktop mode), or
`LSEG_USERNAME` / `LSEG_PASSWORD` / `LSEG_APP_KEY` in `.env` (cloud platform mode).

---
**Output artifacts (gitignored — regenerate from this notebook):**
- `data/raw/news/refinitiv/{ticker}_news_2003_2014.parquet` — full article dataset
- `data/raw/news/refinitiv/{ticker}_daily_counts_2003_2014.csv` — article count per calendar day
- `data/raw/news/refinitiv/{ticker}_coverage_summary.json` — weekly coverage stats
- `data/raw/news/refinitiv/{ticker}_story_text_checkpoint.parquet` — in-progress checkpoint (safe to delete after the run completes)

In [ ]:
from __future__ import annotations

import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv

try:
    from tqdm.auto import tqdm
except ImportError:
    raise ImportError("Install tqdm: pip install tqdm") from None

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env")

from sentiment_ltr.data.news_coverage import (
    build_news_query,
    daily_article_counts,
    fetch_headlines_for_window,
    month_chunks,
    summarize_news_coverage,
    weekly_article_counts,
)
from sentiment_ltr.data.refinitiv_queries import ticker_to_ric_candidates
from sentiment_ltr.data.refinitiv_session import open_refinitiv_session

print(f"Project root: {PROJECT_ROOT}")

## Parameters — change `TICKER` to run for any other stock

In [ ]:
TICKER = "AAPL"
START  = "2003-01-01"
END    = "2014-12-31"

# Checkpoint cadence: save after every N story-text fetches so a crash is resumable.
CHECKPOINT_EVERY = 50

# Delay between story-text API calls (seconds). Increase if you hit rate limits.
STORY_FETCH_DELAY = 0.25

NEWS_DIR   = PROJECT_ROOT / "data" / "raw" / "news" / "refinitiv"
NEWS_DIR.mkdir(parents=True, exist_ok=True)

ARTICLES_PATH   = NEWS_DIR / f"{TICKER.lower()}_news_2003_2014.parquet"
DAILY_PATH      = NEWS_DIR / f"{TICKER.lower()}_daily_counts_2003_2014.csv"
SUMMARY_PATH    = NEWS_DIR / f"{TICKER.lower()}_coverage_summary.json"
CHECKPOINT_PATH = NEWS_DIR / f"{TICKER.lower()}_story_text_checkpoint.parquet"

print(f"Ticker  : {TICKER}")
print(f"Window  : {START} → {END}")
print(f"Output  : {ARTICLES_PATH}")

## Step 1 — Fetch All Headlines

Uses month-by-month chunking to stay within Refinitiv's per-request pagination limits.
Expect this to take 2–5 minutes for a 12-year window.

In [ ]:
import lseg.data as ld

open_refinitiv_session(PROJECT_ROOT, ld)
print("Refinitiv session opened.")

In [ ]:
chunks = month_chunks(START, END)  # one entry per calendar month, e.g. 144 for 2003-2014

headlines: pd.DataFrame | None = None
ric: str | None = None
fetch_errors: list[str] = []

for ric_candidate in ticker_to_ric_candidates(TICKER):
    query = build_news_query(ric_candidate)
    chunk_frames: list[pd.DataFrame] = []

    with tqdm(chunks, desc=f"{TICKER} ({ric_candidate})", unit="month",
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} months  [{elapsed}<{remaining}]") as pbar:
        for chunk_start, chunk_end in pbar:
            month_label = chunk_start.strftime("%Y-%m")
            pbar.set_postfix_str(month_label)
            try:
                frame = fetch_headlines_for_window(
                    ld, query,
                    chunk_start.strftime("%Y-%m-%d"),
                    chunk_end.strftime("%Y-%m-%d"),
                )
                if not frame.empty:
                    chunk_frames.append(frame)
            except Exception as exc:
                fetch_errors.append(f"{ric_candidate} {month_label}: {exc}")
                pbar.set_postfix_str(f"{month_label} ERROR")

    if not chunk_frames:
        fetch_errors.append(f"{ric_candidate}: no headlines in any month")
        continue

    combined = pd.concat(chunk_frames, ignore_index=True)
    combined = (
        combined.drop_duplicates(subset=["storyId"], keep="first")
        if "storyId" in combined.columns
        else combined.drop_duplicates(subset=["article_time", "headline"], keep="first")
    )

    if not combined.empty:
        headlines = combined.sort_values("article_time")
        ric = ric_candidate
        break

if headlines is None or ric is None:
    raise ValueError(f"No headlines found for {TICKER}. Errors:\n" + "\n".join(fetch_errors))

if fetch_errors:
    print(f"Non-fatal fetch errors ({len(fetch_errors)}):")
    for e in fetch_errors[:5]:
        print(f"  {e}")

print(f"\nRIC resolved    : {ric}")
print(f"Total rows      : {len(headlines):,}")
if "storyId" in headlines.columns:
    print(f"Unique storyIds : {headlines['storyId'].nunique():,}")
headlines.head()

## Step 2 — Coverage Statistics

Compute the daily and weekly article counts and check whether this ticker passes the paper's
news filter (average ≥ 1 article per week over 2003–2014).

In [ ]:
daily   = daily_article_counts(headlines, START, END)
weekly  = weekly_article_counts(daily)
summary = summarize_news_coverage(TICKER, ric, headlines, START, END)

print(f"Total articles (unique storyIds) : {summary.total_articles:,}")
print(f"Calendar days in range           : {summary.calendar_days_in_range:,}")
print(f"Calendar days with ≥1 article    : {summary.calendar_days_with_news:,}")
print(f"Weeks in range                   : {summary.weeks_in_range:,}")
print(f"Weeks with zero articles         : {summary.weeks_with_zero_articles:,}")
print(f"Avg articles / week              : {summary.avg_articles_per_week:.2f}")
print(f"Passes paper threshold (≥1/wk)   : {summary.passes_paper_weekly_threshold}")

In [ ]:
nonzero = daily[daily["article_count"] > 0].copy()
fig = px.bar(
    nonzero,
    x="date",
    y="article_count",
    title=f"{TICKER} — Refinitiv articles per day, {START} to {END}",
    labels={"date": "Date", "article_count": "Articles"},
    color_discrete_sequence=["#4C78A8"],
)
fig.update_layout(hovermode="closest", height=420)
fig.show()

In [ ]:
weekly_nonzero = weekly[weekly["article_count"] > 0].copy()
fig2 = px.line(
    weekly_nonzero,
    x="week_start",
    y="article_count",
    title=f"{TICKER} — Weekly article count, {START} to {END}",
    labels={"week_start": "Week", "article_count": "Articles"},
    color_discrete_sequence=["#4C78A8"],
)
fig2.update_traces(mode="lines+markers", marker={"size": 3})
fig2.update_layout(hovermode="closest", height=420)
fig2.show()

## Step 3 — Fetch Full Article Text

For every headline with a `storyId`, call `ld.news.get_story()` to retrieve the full article body.

- A checkpoint is saved every `CHECKPOINT_EVERY` fetches so this cell can be re-run after a crash without starting over.
- Stories that return an error are recorded with `article_text = None`.
- The Refinitiv session opened in Step 1 is reused throughout.

In [ ]:
if "storyId" not in headlines.columns:
    raise ValueError(
        "Headlines DataFrame has no storyId column. "
        "Cannot fetch full article text without storyIds."
    )

to_fetch = headlines.dropna(subset=["storyId"]).copy()
to_fetch = to_fetch.drop_duplicates(subset=["storyId"], keep="first")
print(f"Unique storyIds to fetch: {len(to_fetch):,}")

# Load checkpoint if it exists so we skip already-fetched stories.
if CHECKPOINT_PATH.exists():
    checkpoint = pd.read_parquet(CHECKPOINT_PATH)
    already_done = set(checkpoint["storyId"].dropna().astype(str))
    print(f"Resuming from checkpoint: {len(already_done):,} stories already fetched.")
else:
    checkpoint = pd.DataFrame()
    already_done = set()

pending = to_fetch[~to_fetch["storyId"].astype(str).isin(already_done)].copy()
print(f"Remaining: {len(pending):,} stories to fetch.")

In [ ]:
fetched_rows: list[dict] = []
errors_count = 0

with tqdm(pending.iterrows(), total=len(pending), unit="story",
          bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} stories  [{elapsed}<{remaining}, {rate_fmt}]") as pbar:
    for i, (_, row) in enumerate(pbar, start=1):
        story_id = str(row["storyId"]).strip()
        text = None
        try:
            raw = ld.news.get_story(story_id, format=ld.news.Format.TEXT)
            text = str(raw) if raw is not None else None
        except Exception as exc:
            errors_count += 1
            if errors_count <= 3:
                pbar.write(f"  [warn] storyId {story_id}: {exc}")

        fetched_rows.append({"storyId": story_id, "article_text": text})
        pbar.set_postfix(errors=errors_count)

        if i % CHECKPOINT_EVERY == 0 or i == len(pending):
            batch = pd.DataFrame(fetched_rows)
            combined = pd.concat([checkpoint, batch], ignore_index=True) if not checkpoint.empty else batch
            combined.to_parquet(CHECKPOINT_PATH, index=False)
            checkpoint = combined
            fetched_rows = []
            pbar.write(f"  checkpoint saved at {i:,}/{len(pending):,} ({errors_count} errors)")

        if STORY_FETCH_DELAY > 0:
            time.sleep(STORY_FETCH_DELAY)

print(f"\nDone. {len(pending):,} stories processed, {errors_count:,} fetch errors.")

## Step 4 — Assemble the Final Dataset

Join the fetched story text back to the full headlines table.
Schema: one row per article with date, headline, story text, source, and coverage counts.

In [ ]:
text_df = pd.read_parquet(CHECKPOINT_PATH)[["storyId", "article_text"]].copy()
text_df["storyId"] = text_df["storyId"].astype(str)
text_df = text_df.drop_duplicates(subset=["storyId"], keep="first")

articles = headlines.copy()
articles["storyId"] = articles["storyId"].astype(str)
articles = articles.merge(text_df, on="storyId", how="left")

# Attach daily count so every row knows how many articles ran that day.
daily_lookup = daily.rename(columns={"article_count": "articles_that_day"}).copy()
daily_lookup["date"] = pd.to_datetime(daily_lookup["date"]).dt.normalize()
articles["article_date"] = pd.to_datetime(articles["article_date"]).dt.normalize()
articles = articles.merge(daily_lookup, left_on="article_date", right_on="date", how="left").drop(columns=["date"])

articles["ticker"] = TICKER.upper()
articles["ric"]    = ric

col_order = [
    "ticker", "ric",
    "article_date", "article_time",
    "headline", "storyId", "sourceCode",
    "articles_that_day",
    "article_text",
]
articles = articles[[c for c in col_order if c in articles.columns]].sort_values("article_time")

print(f"Final dataset: {len(articles):,} rows")
print(f"Rows with text: {articles['article_text'].notna().sum():,}")
print(f"Rows missing text: {articles['article_text'].isna().sum():,}")
articles.head()

## Step 5 — Save Outputs

In [ ]:
# Full article dataset (Parquet — handles long text better than CSV).
articles.to_parquet(ARTICLES_PATH, index=False)

# Daily article counts (small, CSV is fine).
daily.to_csv(DAILY_PATH, index=False)

# Coverage summary JSON.
summary_dict = {
    "created_at":                   datetime.now(timezone.utc).isoformat(),
    "ticker":                       summary.ticker,
    "ric":                          summary.ric,
    "start_date":                   summary.start_date,
    "end_date":                     summary.end_date,
    "total_articles":               summary.total_articles,
    "calendar_days_with_news":      summary.calendar_days_with_news,
    "calendar_days_in_range":       summary.calendar_days_in_range,
    "avg_articles_per_calendar_day": round(summary.avg_articles_per_calendar_day, 4),
    "avg_articles_per_week":        round(summary.avg_articles_per_week, 4),
    "weeks_in_range":               summary.weeks_in_range,
    "weeks_with_zero_articles":     summary.weeks_with_zero_articles,
    "passes_paper_weekly_threshold": summary.passes_paper_weekly_threshold,
    "articles_with_text":           int(articles["article_text"].notna().sum()),
    "articles_missing_text":        int(articles["article_text"].isna().sum()),
    "output_parquet":               str(ARTICLES_PATH.relative_to(PROJECT_ROOT)),
    "output_daily_csv":             str(DAILY_PATH.relative_to(PROJECT_ROOT)),
}
SUMMARY_PATH.write_text(json.dumps(summary_dict, indent=2) + "\n", encoding="utf-8")

print(f"Saved articles   → {ARTICLES_PATH}")
print(f"Saved daily CSV  → {DAILY_PATH}")
print(f"Saved summary    → {SUMMARY_PATH}")

print()
print("To load in another notebook:")
print(f"  articles = pd.read_parquet(PROJECT_ROOT / 'data' / 'raw' / 'news' / 'refinitiv' / '{TICKER.lower()}_news_2003_2014.parquet')")

## Step 6 — Close Session

In [ ]:
try:
    ld.close_session()
    print("Refinitiv session closed.")
except Exception as exc:
    print(f"Session close warning (safe to ignore): {exc}")